In [ ]:
pip install requests pandas sqlalchemy


In [ ]:
import requests
import pandas as pd
import math
import time

API_URL = "http://YOUR_HOST:PORT/api.php"
HEADERS = {"X-API-Key": "YOUR_API_KEY"}

QARAJ_LAT = 0.0000
QARAJ_LON = 0.0000
RADIUS_METR = 100

def haversine_merator(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    a = math.sin((lat2 - lat1) / 2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin((lon2 - lon1) / 2)**2
    return 2 * math.asin(math.sqrt(a)) * 6371000

def get_all_vehicles():
    params = {"endpoint": "vehicles"}
    try:
        response = requests.get(API_URL, headers=HEADERS, params=params, timeout=60)
        if response.status_code == 200:
            data = response.json()
            if data.get("success"):
                vehicles_data = data.get("data", [])
                if vehicles_data and isinstance(vehicles_data[0], dict):
                    return [v.get("vehicle_id") for v in vehicles_data if "vehicle_id" in v]
                else:
                    return vehicles_data
    except Exception as e:
        print("Maşınların siyahısını çəkərkən xəta baş verdi:", e)
    return []

def fetch_telemetry_data(vehicle_id, start_date, end_date):
    raw_points = []
    limit = 5000
    offset = 0
    retries = 0

    while True:
        params = {
            "endpoint": "history",
            "vehicle_id": vehicle_id,
            "from": start_date,
            "to": end_date,
            "limit": limit,
            "offset": offset
        }

        try:
            response = requests.get(API_URL, headers=HEADERS, params=params, timeout=120)
        except requests.exceptions.RequestException as e:
            print(f"   [!] Server xətası (Timeout/Connection): Maşın {vehicle_id}, Offset {offset}. Bu hissə atlanır.")
            break

        if response.status_code == 200:
            data = response.json()
            if data.get("success"):
                records = data.get("data", [])
                total = data.get("total", 0)
                count = data.get("count", 0)

                for row in records:
                    try:
                        speed = float(row.get("speed", -1))
                    except (ValueError, TypeError):
                        speed = -1

                    lat = row.get("latitude")
                    lon = row.get("longitude")

                    if lat is not None and lon is not None:
                        try:
                            lat_f = float(lat)
                            lon_f = float(lon)

                            if lat_f != 0.0 and lon_f != 0.0:
                                try:
                                    distance = float(row.get("distance", 0))
                                except (ValueError, TypeError):
                                    distance = 0.0

                                raw_points.append({
                                    "vehicle_id": row.get("vehicle_id"),
                                    "timestamp": row.get("timestamp"),
                                    "latitude": lat_f,
                                    "longitude": lon_f,
                                    "speed": speed,
                                    "distance": distance
                                })
                        except (ValueError, TypeError):
                            pass

                if offset + count >= total or count == 0:
                    break

                offset += count
            else:
                break
        else:
            print(f"   [!] Server xəta kodu qaytardı: {response.status_code}")
            break

    df = pd.DataFrame(raw_points)

    if not df.empty:
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df = df.sort_values('timestamp').reset_index(drop=True)

        df['speed'] = df['speed'].apply(lambda x: x if x > 0 else 0).round(1)
        df['latitude'] = df['latitude'].round(6)
        df['longitude'] = df['longitude'].round(6)

        df['time_group'] = df['timestamp'].dt.floor('5min')
        df = df.drop_duplicates(subset=['vehicle_id', 'time_group'], keep='first').copy()

        df['distance'] = df['distance'].replace(0.0, float('nan')).ffill().fillna(0)

        df['time_diff_hr'] = df['timestamp'].diff().dt.total_seconds() / 3600.0
        df['dist_diff'] = df['distance'].diff()
        df['implied_speed'] = df['dist_diff'] / df['time_diff_hr']

        df = df[(df['implied_speed'] <= 150) | (df['implied_speed'].isna())].copy()

        df['dist_to_garage'] = df.apply(lambda r: haversine_merator(r['latitude'], r['longitude'], QARAJ_LAT, QARAJ_LON), axis=1)

        def determine_status(r):
            if r['speed'] > 0:
                return 'Hərəkətdə'
            elif r['dist_to_garage'] <= RADIUS_METR:
                return 'Qarajda'
            else:
                return 'Digər ərazi (Dayanma)'

        df['status'] = df.apply(determine_status, axis=1)
        df.loc[df['time_diff_hr'] > 2, 'status'] = 'Bilinməyən ərazi (Siqnal qopması)'

        df = df.drop(columns=['time_diff_hr', 'dist_diff', 'implied_speed', 'dist_to_garage', 'time_group'])

    return df

start_time = "2026-01-01 00:00:00"
end_time = "2026-08-30 23:59:59"

all_vehicles = get_all_vehicles()
print(f"Bazada ümumi {len(all_vehicles)} maşın tapıldı. Xam datanın yığılması başlayır...\n")

all_stops_dataframes = []

for idx, v_id in enumerate(all_vehicles, start=1):
    print(f"[{idx}/{len(all_vehicles)}] {v_id} nömrəli maşın yoxlanılır...")
    df = fetch_telemetry_data(v_id, start_time, end_time)

    if not df.empty:
        all_stops_dataframes.append(df)

    time.sleep(1)

if all_stops_dataframes:
    final_df = pd.concat(all_stops_dataframes, ignore_index=True)
    filename = "butun_masinlar_ideal_xam_data_SON_SON_SON.csv"

    final_df.to_csv(filename, index=False, sep=';', decimal=',', encoding='utf-8-sig')

    print(f"\nƏməliyyat tam yekunlaşdı! Ümumi {len(final_df)} ədəd qeyd tapıldı.")
    print(f"Bütün məlumatlar '{filename}' faylına yadda saxlanıldı.")
else:
    print("\nGöstərilən tarix aralığında heç bir maşın üçün qeyd tapılmadı.")


In [ ]:
import requests
import pandas as pd
import time
from datetime import datetime

API_URL = "http://YOUR_HOST:PORT/api.php"
HEADERS = {"X-API-Key": "YOUR_API_KEY"}

def get_all_vehicles():
    params = {"endpoint": "vehicles"}
    try:
        response = requests.get(API_URL, headers=HEADERS, params=params, timeout=60)
        if response.status_code == 200:
            data = response.json()
            if data.get("success"):
                vehicles_data = data.get("data", [])
                if vehicles_data and isinstance(vehicles_data[0], dict):
                    return [v.get("vehicle_id") for v in vehicles_data if "vehicle_id" in v]
                else:
                    return vehicles_data
    except Exception as e:
        print("Maşınların siyahısını çəkərkən xəta baş verdi:", e)
    return []

def safe_float(value, default=0.0):
    try:
        if value is None or value == "":
            return default
        return float(value)
    except (ValueError, TypeError):
        return default

def fetch_telemetry_data(vehicle_id, start_date, end_date):
    raw_points = []
    seen_time_groups = set()

    limit = 5000
    offset = 0

    while True:
        params = {
            "endpoint": "history",
            "vehicle_id": vehicle_id,
            "from": start_date,
            "to": end_date,
            "limit": limit,
            "offset": offset
        }

        try:
            response = requests.get(API_URL, headers=HEADERS, params=params, timeout=120)
        except requests.exceptions.RequestException as e:
            print(f"   [!] Server xətası (Timeout): Maşın {vehicle_id}, Offset {offset}. Atlanır.")
            break

        if response.status_code == 200:
            data = response.json()
            if data.get("success"):
                records = data.get("data", [])
                total = data.get("total", 0)
                count = data.get("count", 0)

                for row in records:
                    timestamp_str = row.get("timestamp")
                    lat = row.get("latitude")
                    lon = row.get("longitude")

                    if timestamp_str and lat is not None and lon is not None:
                        lat_f = safe_float(lat)
                        lon_f = safe_float(lon)

                        if lat_f != 0.0 and lon_f != 0.0:
                            try:
                                dt_obj = datetime.strptime(timestamp_str, "%Y-%m-%d %H:%M:%S")

                                minute_floor = (dt_obj.minute // 5) * 5
                                time_group = dt_obj.replace(minute=minute_floor, second=0, microsecond=0)
                            except ValueError:
                                continue

                            if time_group in seen_time_groups:
                                continue

                            seen_time_groups.add(time_group)

                            raw_points.append({
                                "vehicle_id": row.get("vehicle_id"),
                                "timestamp": timestamp_str,
                                "latitude": lat_f,
                                "longitude": lon_f,
                                "speed": safe_float(row.get("speed")),
                                "speed2": safe_float(row.get("speed2")),
                                "engine_speed": safe_float(row.get("engine_speed")),
                                "distance": safe_float(row.get("distance"))
                            })

                if offset + count >= total or count == 0:
                    break
                offset += count
            else:
                break
        else:
            print(f"   [!] Server xəta kodu qaytardı: {response.status_code}")
            break

    df = pd.DataFrame(raw_points)

    if not df.empty:
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df = df.sort_values('timestamp').reset_index(drop=True)

    return df

start_time = "2026-01-01 00:00:00"
end_time = "2026-08-30 23:59:59"

all_vehicles = get_all_vehicles()
print(f"Bazada ümumi {len(all_vehicles)} maşın tapıldı. Xam datanın yığılması başlayır...\n")

all_stops_dataframes = []

for idx, v_id in enumerate(all_vehicles, start=1):
    print(f"[{idx}/{len(all_vehicles)}] {v_id} nömrəli maşın yoxlanılır...")
    df = fetch_telemetry_data(v_id, start_time, end_time)

    if not df.empty:
        all_stops_dataframes.append(df)

    time.sleep(1)

if all_stops_dataframes:
    final_df = pd.concat(all_stops_dataframes, ignore_index=True)
    filename = "MTrans_Xalis_Baza_2026.csv"

    final_df.to_csv(filename, index=False, sep=';', decimal=',', encoding='utf-8-sig')

    print(f"\nƏməliyyat tam yekunlaşdı! Ümumi {len(final_df)} ədəd qeyd tapıldı.")
    print(f"Bütün məlumatlar '{filename}' faylına yadda saxlanıldı.")
else:
    print("\nGöstərilən tarix aralığında heç bir maşın üçün qeyd tapılmadı.")
